# 01 — Exploratory Data Analysis

This notebook answers three questions:
1. What does the data look like? (distributions, missing values)
2. What drives home prices in Saratoga Springs & Eagle Mountain?
3. Is there a real price difference between the two cities — and do they offer different value?

**Before running this notebook:**
- Option A (Quick): Run `python main.py --run-now --no-email` from the project root first — this fetches listings automatically.
- Option B (Manual): Download CSVs from redfin.com and save to `../data/raw/`

Then re-run all cells.

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.scraper import load_all_raw_csv
from src.features import prepare_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.float_format', '${:,.0f}'.format)

## 1. Load Data

In [ ]:
raw = load_all_raw_csv('../data/raw')
print(f'Raw rows: {len(raw):,}  |  Columns: {raw.shape[1]}')
raw.head(3)

In [ ]:
df = prepare_dataset(raw)
print(f'Prepared rows: {len(df):,}  |  Columns: {df.shape[1]}')
df[['city', 'price', 'beds', 'baths', 'sqft', 'lot_sqft', 'year_built', 'days_on_market', 'sold']].describe()

## 2. Missing Value Audit

In [ ]:
core_cols = ['price', 'beds', 'baths', 'sqft', 'lot_sqft', 'year_built',
             'days_on_market', 'hoa_monthly', 'latitude', 'longitude']
missing = df[core_cols].isna().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 4))
missing.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% Missing')
ax.set_title('Missing Value Rate by Column')
for bar, val in zip(ax.patches, missing.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Price Distribution

In [ ]:
active = df[df['sold'] == False].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(active['price'].dropna() / 1e6, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Price ($M)')
axes[0].set_ylabel('Count')
axes[0].set_title('Active Listing Price Distribution')
axes[0].axvline(active['price'].median() / 1e6, color='red', linestyle='--', label=f"Median: ${active['price'].median()/1e6:.2f}M")
axes[0].legend()

# By city
target_cities = ['Saratoga Springs', 'Eagle Mountain']
city_data = [active[active['city'] == c]['price'].dropna() / 1e6 for c in target_cities]
bp = axes[1].boxplot(city_data, labels=target_cities, patch_artist=True)
colors = ['#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Price ($M)')
axes[1].set_title('Price by City — Active Listings')

plt.tight_layout()
plt.show()

# Summary stats by city
print(active.groupby('city')['price'].describe().applymap(lambda x: f'${x:,.0f}' if isinstance(x, float) else x))

## 4. Price Per Sqft — The Real Value Metric

Price per sqft is a more apples-to-apples comparison across differently sized homes.

In [ ]:
active['price_per_sqft_calc'] = active['price'] / active['sqft']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(active['price_per_sqft_calc'].dropna(), bins=40, color='seagreen', edgecolor='white')
axes[0].set_xlabel('$/sqft')
axes[0].set_title('Price Per Sqft Distribution')
med = active['price_per_sqft_calc'].median()
axes[0].axvline(med, color='red', linestyle='--', label=f'Median: ${med:.0f}/sqft')
axes[0].legend()

# By city
for city, color in zip(target_cities, colors):
    subset = active[active['city'] == city]['price_per_sqft_calc'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, edgecolor='white', label=city)
axes[1].set_xlabel('$/sqft')
axes[1].set_title('$/sqft Distribution by City')
axes[1].legend()

plt.tight_layout()
plt.show()

# Median $/sqft comparison — the moment of truth!
comparison = active.groupby('city')['price_per_sqft_calc'].agg(['median', 'mean', 'count'])
comparison.columns = ['Median $/sqft', 'Mean $/sqft', 'Count']
print("\n$/sqft comparison:")
print(comparison)

## 5. Feature Correlations with Price

In [ ]:
num_cols = ['price', 'beds', 'baths', 'sqft', 'lot_sqft', 'year_built',
            'days_on_market', 'hoa_monthly', 'feat_home_age', 'feat_sqft_per_bed']
num_cols = [c for c in num_cols if c in active.columns]

corr = active[num_cols].corr()['price'].drop('price').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
corr.plot(kind='barh', ax=ax, color=corr.map(lambda v: '#27ae60' if v > 0 else '#e74c3c'))
ax.set_xlabel('Pearson Correlation with Price')
ax.set_title('Which Features Correlate Most with Price?')
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(ax.patches, corr.values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Scatter: Sqft vs Price

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for city, color in zip(target_cities, colors):
    subset = active[(active['city'] == city) & active['sqft'].notna() & active['price'].notna()]
    ax.scatter(subset['sqft'], subset['price'] / 1e3, alpha=0.5, label=city, color=color, s=40)

# Regression line
plot_data = active[active['sqft'].notna() & active['price'].notna()]
m, b = np.polyfit(plot_data['sqft'], plot_data['price'] / 1e3, 1)
x_range = np.linspace(plot_data['sqft'].min(), plot_data['sqft'].max(), 100)
ax.plot(x_range, m * x_range + b, 'k--', linewidth=1.5, label=f'Trend (${m*1000:.0f}/sqft)')

ax.set_xlabel('Square Feet')
ax.set_ylabel('Price ($K)')
ax.set_title('Square Feet vs Price — Active Listings')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Year Built vs Price — Does Newer = More Expensive?

In [ ]:
yr_data = active[active['year_built'].notna() & active['price'].notna()]
yr_bins = pd.cut(yr_data['year_built'], bins=range(1980, 2030, 5))
yr_group = yr_data.groupby(yr_bins)['price'].median() / 1e3

fig, ax = plt.subplots(figsize=(12, 5))
yr_group.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_xlabel('Year Built (5-year bins)')
ax.set_ylabel('Median Price ($K)')
ax.set_title('Median Price by Year Built')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Map View — Geographic Price Distribution

In [ ]:
map_data = active[active['latitude'].notna() & active['longitude'].notna() & active['price'].notna()].copy()

if len(map_data) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.scatter(
        map_data['longitude'],
        map_data['latitude'],
        c=map_data['price'] / 1e3,
        cmap='RdYlGn_r',
        s=60,
        alpha=0.7,
        edgecolors='white',
        linewidths=0.5,
    )
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Price ($K)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Geographic Price Distribution (Red = Expensive, Green = Cheaper)')
    plt.tight_layout()
    plt.show()
else:
    print('No latitude/longitude data available — skipping map.')

## 9. Days on Market — How Fast Does Each City Move?

In [ ]:
dom_data = active[active['days_on_market'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(dom_data['days_on_market'].clip(0, 180), bins=30, color='coral', edgecolor='white')
axes[0].set_xlabel('Days on Market')
axes[0].set_ylabel('Count')
axes[0].set_title('Days on Market Distribution (capped at 180)')
axes[0].axvline(dom_data['days_on_market'].median(), color='darkred', linestyle='--',
                label=f"Median: {dom_data['days_on_market'].median():.0f} days")
axes[0].legend()

# By city
for city, color in zip(target_cities, colors):
    subset = dom_data[dom_data['city'] == city]['days_on_market'].clip(0, 180)
    if len(subset) > 0:
        axes[1].hist(subset, bins=20, alpha=0.6, color=color, edgecolor='white', label=city)
axes[1].set_xlabel('Days on Market')
axes[1].set_title('Days on Market by City')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Median days on market by city:")
print(active.groupby('city')['days_on_market'].median())

## 10. Summary Insights

Fill in after running the cells above:

In [ ]:
print("=" * 60)
print("KEY STATS SUMMARY")
print("=" * 60)

for city in target_cities:
    subset = active[active['city'] == city]
    if len(subset) == 0:
        print(f"\n{city}: No data")
        continue
    print(f"\n{city} ({len(subset)} active listings):")
    print(f"  Median price:       ${subset['price'].median():,.0f}")
    if 'price_per_sqft_calc' in subset.columns:
        print(f"  Median $/sqft:      ${subset['price_per_sqft_calc'].median():.0f}")
    print(f"  Median sqft:        {subset['sqft'].median():,.0f}")
    print(f"  Median beds:        {subset['beds'].median():.0f}")
    print(f"  Median baths:       {subset['baths'].median():.1f}")
    print(f"  Median yr built:    {subset['year_built'].median():.0f}")
    print(f"  Median days on mkt: {subset['days_on_market'].median():.0f}")